# Detect case patterns in the customer-complaints example with `itk-patterns`

This notebook uses **`packages/itk-patterns`** (`import itk_patterns`), not the legacy
`intelligence_toolkit.detect_case_patterns` module or the Streamlit workflow. It reads one
existing repository input directly, without copying it or generating new data:

- `example_outputs/detect_case_patterns/customer_complaints/customer_complaints_prepared.csv`

That file has 3,000 customer-complaint records with eight attributes (`city`, `age_range`,
five `*_issue` flags, `product_code`) and a half-year `period`. The notebook builds the
same attribute model the workflow's "Prepare case data" tab builds, embeds one attribute
graph per period, detects attribute patterns whose records converge in a period, ranks
them, re-runs with different thresholds without re-embedding, shows the time series and
attribute counts behind one pattern, and asks an LLM for the pattern report when an API
key is configured.

Everything up to the report is deterministic and offline: the graph encoder embedding has
no random component, so re-running gives the same frames. The report step is shown with
the exact prompt it would send, and skipped unless `OPENAI_API_KEY` (or an Azure OpenAI
endpoint) is set in the environment.

## 1. Install and select a Python kernel

Use **Python 3.11 or 3.12**. The package needs `itk-core` (for the prompt helpers and the
OpenAI client), `networkx`, `numpy`, `polars`, `pydantic` and `scipy`. From the repository
root, run once in a terminal:

```powershell
uv venv --python 3.12 .venv-patterns-demo
uv pip install --python .venv-patterns-demo\Scripts\python.exe -e .\packages\itk-core -e .\packages\itk-patterns plotly ipykernel
.\.venv-patterns-demo\Scripts\python.exe -m ipykernel install --user --name itk-patterns-demo --display-name "Python (itk-patterns demo)"
```

On macOS/Linux use `.venv-patterns-demo/bin/python` and slash-separated paths. Install
**both local packages together**: `itk-core` on PyPI is a different project. If you already
use the repository's own environment (`uv sync --extra dev`), its kernel works too, since the
workspace installs `itk-patterns` in development mode.

Open this notebook in VS Code or Jupyter, select the kernel, and run all cells. Input lookup
walks up from the working directory, so the notebook runs from the repository root or from
its own folder. `plotly` is optional: the chart cell skips itself when it is not installed,
and the chart is drawn from a frame shown just above it.

In [ ]:
# Copyright (c) 2026 Microsoft Corporation. All rights reserved.
# Licensed under the MIT license. See LICENSE file in the project.

import os
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import polars as pl
from IPython.display import Markdown
from IPython.display import display as show
from itk_patterns import DetectionConfig, PatternConfig, PatternDetector
from itk_patterns.models import ALL_PERIODS

try:
    import plotly.express as px
except ImportError:  # the chart is optional
    px = None

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(80)

cwd = Path.cwd().resolve()
relative_input = (
    Path("example_outputs")
    / "detect_case_patterns"
    / "customer_complaints"
    / "customer_complaints_prepared.csv"
)
repo_root = next(
    (parent for parent in (cwd, *cwd.parents) if (parent / relative_input).is_file()),
    None,
)
if repo_root is None:
    message = "Run from within the repository containing example_outputs/."
    raise FileNotFoundError(message)
input_path = repo_root / relative_input

for name in ("itk-patterns", "itk-core", "networkx", "numpy", "polars", "scipy"):
    try:
        print(f"{name} {version(name)}")
    except PackageNotFoundError:
        print(f"{name} (not installed as a distribution)")
print("input:", input_path.relative_to(repo_root).as_posix())

## 2. Prepare the attribute model

`PatternDetector.prepare` casts every column to text (`""` for missing), numbers the
records as subjects, and unpivots them into one link per `(subject, period, attribute)`.
Attribute nodes are named `type=value`, with the `=` separator the Streamlit workflow uses.
Every column except `period` is an attribute here, which is what `PatternConfig` does when
`attribute_columns` is left unset.

The result also carries one weighted co-occurrence graph per period over all 94
attributes. Pairs that appear together in a period are scaled onto
`[min_edge_weight, 1]`; pairs that never appear together get a small
`missing_edge_prop * min_edge_weight` weight so that every graph has the same node set.

In [ ]:
cases = pl.read_csv(input_path, infer_schema_length=0)
config = PatternConfig(period_column="period")
result = PatternDetector.prepare(cases, config)

print(result.summary)
print("periods:", ", ".join(result.periods))
show(Markdown("**First links of the attribute model**"))
show(result.links.head(10))

graph = result.period_graphs["2023-H1"]
edges = pl.DataFrame(
    [(s, t, w) for s, t, w in graph.edges(data="weight")],
    schema={"source": pl.Utf8, "target": pl.Utf8, "weight": pl.Float64},
    orient="row",
).sort("weight", descending=True)
show(
    Markdown(
        f"**Heaviest edges of the 2023-H1 graph ({graph.number_of_edges()} edges)**"
    )
)
show(edges.head(8))

## 3. Embed the period graphs

`PatternDetector.embed` runs the graph encoder embedding on each period graph: a self loop
on every node, symmetric Laplacian normalisation, then each node's row is the weighted
average of its links into each attribute type, row-normalised. Fusing the periods gives
every node a position per period, an all-time centroid (`ALL`) and the centroid of the
periods before each period (`<period`), with the cosine distance each position has moved.

Any object with `embed(graph, nodes, labels, config) -> ndarray` can replace the default
`EncoderEmbedder`; the unit tests use a two-dimensional fake.

In [ ]:
detector = PatternDetector()
result = detector.embed(result)
assert result.positions is not None
assert result.shifts is not None

node = "product_code=A"
positions = result.positions[node]
print(
    f"{node}: {len(positions)} positions, each of {positions[ALL_PERIODS].shape[0]} dimensions"
)

shifts = pl.DataFrame(
    [
        (attribute, period, shift)
        for attribute, track in result.shifts.items()
        for period, shift in track.items()
        if not period.startswith("<")
    ],
    schema={"attribute": pl.Utf8, "period": pl.Utf8, "shift_from_centroid": pl.Float64},
    orient="row",
).sort("shift_from_centroid", descending=True)
show(Markdown("**Attributes that moved furthest from their all-time position**"))
show(shifts.head(10))

## 4. Detect and rank patterns

`PatternDetector.detect` looks, in every period, for pairs of different-type attributes
whose positions converge (closer in that period than their centroids are) and that at
least `min_pattern_count` records carry together. Pairs grow into longer patterns, up to
`max_pattern_length`, as long as every pair inside the pattern converges and strictly more
than `min_pattern_count` records match. Each pattern's count in the period is compared
with its own history across all periods: `z_score` is the standardised excess,
`detections` the number of periods the pattern was found in, and

```
overall_score = normalise(z_score * length * detections * ln(1 + count))
```

The thresholds below are the Streamlit defaults. `patterns_display` applies the legacy
rounding; `result.patterns` keeps the unrounded numbers.

In [ ]:
detection = DetectionConfig(min_pattern_count=100, max_pattern_length=5)
result = PatternDetector.detect(result, detection)

patterns = result.patterns
share = result.close_pairs / result.all_pairs * 100 if result.all_pairs else 0
show(
    Markdown(
        f"Over **{patterns['period'].n_unique()}** periods, detected **{patterns.height}** "
        f"attribute patterns (**{patterns['pattern'].n_unique()}** unique) from "
        f"**{result.close_pairs}**/**{result.all_pairs}** converging attribute pairs "
        f"(**{share:.2f}%**)."
    )
)
show(result.patterns_display.head(15))

## 5. Re-run with other thresholds without re-embedding

The embedding positions stay on the result, so changing the detection thresholds only
re-runs the pair test, the pattern growth and the scoring. Fewer required records and
shorter patterns find more, smaller patterns.

In [ ]:
variants = [
    DetectionConfig(min_pattern_count=100, max_pattern_length=5),
    DetectionConfig(min_pattern_count=50, max_pattern_length=3),
    DetectionConfig(min_pattern_count=150, max_pattern_length=5),
]
rows = []
for variant in variants:
    rerun = PatternDetector.detect(result, variant)
    rows.append((
        variant.min_pattern_count,
        variant.max_pattern_length,
        rerun.close_pairs,
        rerun.patterns.height,
        rerun.patterns["pattern"].n_unique(),
        int(rerun.patterns["length"].max()) if rerun.patterns.height else 0,
    ))
show(
    pl.DataFrame(
        rows,
        schema=[
            "min_pattern_count",
            "max_pattern_length",
            "converging_pairs",
            "patterns",
            "unique_patterns",
            "longest",
        ],
        orient="row",
    )
)

## 6. Time series behind one pattern

Selecting a pattern in the workflow shows how many records matched it in every period.
`result.time_series` holds one such series per distinct detected pattern and
`pattern_series` filters it. The chart is drawn from the frame shown just above it.

In [ ]:
top = result.patterns.row(0, named=True)
selected_pattern, selected_period = top["pattern"], top["period"]
show(Markdown(f"**Pattern: {selected_pattern} ({selected_period})**"))

series = result.pattern_series(selected_pattern)
show(series)

if px is None:
    print("plotly is not installed; skipping the chart.")
else:
    figure = px.line(
        series.to_pandas(),
        x="period",
        y="count",
        markers=True,
        title=f"{selected_pattern} ({selected_period})",
    )
    figure.add_vline(x=selected_period, line_dash="dash")
    figure.show()

## 7. Attribute counts of the matching records

`attribute_counts` filters the prepared records to the selected period and pattern and
counts every attribute value they carry. The pattern's own attributes appear with the
full match count; the rest describe the context of the pattern and feed the report.

In [ ]:
counts = PatternDetector.attribute_counts(result, selected_pattern, selected_period)
show(Markdown(f"**{counts['count'].max()} records match in {selected_period}**"))
show(counts)

## 8. Generate the AI pattern report

The report prompt is built from the pattern, its period, the time series and the attribute
counts as CSV, exactly as the Streamlit workflow does. The messages are shown here so the
step is visible without an API key. When `OPENAI_API_KEY` (or an Azure OpenAI endpoint with
the matching auth settings) is set, `PatternDetector.from_openai` wraps `itk_core`'s client
and the report is generated; otherwise the call is skipped. Any async
`(messages, stream, callbacks) -> str` callable can stand in for the client.

In [ ]:
messages = PatternDetector.report_messages(result, selected_pattern, selected_period)
prompt = messages[0]["content"]
show(
    Markdown(
        f"**Prompt: {len(messages)} message(s), {len(prompt):,} characters.** Its start:"
    )
)
print(prompt[:900] + "\n...")

has_key = bool(os.environ.get("OPENAI_API_KEY")) or bool(
    os.environ.get("AZURE_OPENAI_ENDPOINT")
)
if not has_key:
    print(
        "No OPENAI_API_KEY or AZURE_OPENAI_ENDPOINT in the environment; skipping the AI report."
    )
else:
    from itk_core.openai_configuration import OpenAIConfiguration

    reporter = PatternDetector.from_openai(OpenAIConfiguration())
    report = await reporter.report(result, selected_pattern, selected_period)
    show(Markdown(report))